In [2]:
%load_ext kedro.ipython

[02/03/26 23:51:25] INFO     Using                                                                  ]8;id=889425;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=129230;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/framework/project/__init__.py#269\269]8;;\
                             '/Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-                
                             packages/kedro/framework/project/rich_logging.yml' as logging                         
                             configuration.                                                                        

                    INFO     Registered line magic '%reload_kedro'                                   ]8;id=327795;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=112559;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py#64\64]8;;\

                    INFO     Registered line magic '%load_node'                                      ]8;id=894301;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=545592;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py#66\66]8;;\

                    INFO     Resolved project path as:                                              ]8;id=466871;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=515236;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py#181\181]8;;\
                             /Users/me/ysz-vc/kedrogy/example/mykedro.                                   
                             To set a different path, run '%reload_kedro <project_root>'                           

[02/03/26 23:51:26] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=976298;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=113390;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

[02/03/26 23:51:27] INFO     Kedro project mykedro                                                  ]8;id=877765;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=849500;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py#147\147]8;;\

                    INFO     Defined global variable 'context', 'session', 'catalog' and            ]8;id=812726;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py\__init__.py]8;;\:]8;id=569954;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/ipython/__init__.py#148\148]8;;\
                             'pipelines'                                                                           

In [1]:
from collections.abc import Iterable

from srsly.util import JSONInput
import psycopg
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from datasets import Dataset


In [3]:
ads = catalog.load("model_input")

ads

[02/03/26 23:51:38] INFO     Loading data from model_input (HuggingfaceDataset)...             ]8;id=101043;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/io/data_catalog.py\data_catalog.py]8;;\:]8;id=981583;file:///Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/kedro/io/data_catalog.py#1048\1048]8;;\


DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 3
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 1
    })
})

In [ ]:
from pathlib import Path

from kedro.config import OmegaConfigLoader
from kedro.framework.project import settings

conf_path = str(Path("..") / settings.CONF_SOURCE)
conf_loader = OmegaConfigLoader(conf_source=conf_path)
parameters = conf_loader["parameters"]

parameters


{
    'load_examples_options': {
        'data_table_name': 'all_data',
        'dataset_name': 'news_headlines',
        'id_field': 'id'
    },
    'model_options': {
        'dataset_name': 'news_headlines',
        'labels': ['POS', 'NEG'],
        'data_seed': 123,
        'seed': 123
    }
}

In [7]:
parameters = parameters["model_options"]

parameters

{'dataset_name': 'news_headlines', 'labels': ['POS', 'NEG'], 'data_seed': 123, 'seed': 123}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")


def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)


tokenized_ads = ads.map(preprocess_function, batched=True)


In [12]:
tokenized_ads


DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3
    })
    test: Dataset({
        features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1
    })
})

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)


def make_id2label_label2id(lst):
    id2label = {}
    label2id = {}
    for i, label in enumerate(lst):
        id2label[str(i + 1)] = label
        label2id[label] = str(i + 1)
    return {**id2label, **{"0": "__label__OTHER"}}, {
        **label2id,
        **{"__label__OTHER": "0"},
    }


# id2label.json
# id2label = {"0": "__label__OTHER", "1": "ADS"}
# label2id = {"__label__OTHER": 0, "ADS": 1}
id2label, label2id = make_id2label_label2id(parameters["labels"])

id2label, label2id


({'1': 'POS', '2': 'NEG', '0': '__label__OTHER'}, {'POS': '1', 'NEG': '2', '__label__OTHER': '0'})

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-multilingual-uncased",
    num_labels=len(parameters["labels"]) + 1,  # and __label__OTHER
    id2label=id2label,
    label2id=label2id,
)

training_args = TrainingArguments(
    output_dir="data/06_models/train",
    learning_rate=2e-5,
    # per_device_train_batch_size=16,
    # per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # eval_strategy="epoch",
    # save_strategy="best",#"epoch",
    # save_total_limit=1,
    # load_best_model_at_end=True,
    # https://discuss.huggingface.co/t/offline-transformers-trainer/27703/3
    push_to_hub=False,  # otherwise: 401 Client Error: Unauthorized for url: https://huggingface.co/api/repos/create
    data_seed=parameters["data_seed"],  # 123,
    seed=parameters["seed"],  # 123,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ads["train"],
    eval_dataset=tokenized_ads["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# https://clear.ml/docs/latest/docs/guides/frameworks/huggingface/transformers/
trainer.train()


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/transformers/token │
│ ization_utils_base.py:795 in convert_to_tensors                                                  │
│                                                                                                  │
│    792 │   │   │   │   │   value = [value]                                                       │
│    793 │   │   │   │                                                                             │
│    794 │   │   │   │   if not is_tensor(value):                                                  │
│ ❱  795 │   │   │   │   │   tensor = as_tensor(value)                                             │
│    796 │   │   │   │   │                                                                         │
│    797 │   │   │   │   │   # Removing this for now in favor of controlling the shape with `prep  │
│    798 │   │   │   │   │   # # at-least2d                                                        │
│                                                                                                  │
│ /Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/transformers/token │
│ ization_utils_base.py:742 in as_tensor                                                           │
│                                                                                                  │
│    739 │   │   │   │   │   return torch.from_numpy(np.array(value))                              │
│    740 │   │   │   │   if len(flatten(value)) == 0 and dtype is None:                            │
│    741 │   │   │   │   │   dtype = torch.int64                                                   │
│ ❱  742 │   │   │   │   return torch.tensor(value, dtype=dtype)                                   │
│    743 │   │   │                                                                                 │
│    744 │   │   │   is_tensor = torch.is_tensor                                                   │
│    745                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ValueError: too many dimensions 'str'

The above exception was the direct cause of the following exception:

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 │   compute_metrics=compute_metrics,                                                        │
│    9 )                                                                                           │
│   10 # https://clear.ml/docs/latest/docs/guides/frameworks/huggingface/transformers/             │
│ ❱ 11 trainer.train()                                                                             │
│   12                                                                                             │
│                                                                                                  │
│ /Users/me/ysz-vc/kedrogy/example/.venv/lib/python3.12/site-packages/transformers/train │
│ er.py:2325 in train                                                                              │
│                                                                                                  │
│   2322 │   │   │   finally:                                                                      │
│   2323 │   │   │   │   hf_hub_utils.enable_progress_bars()                                       │
│   2324 │   │   else:                                                                             │
│ ❱ 2325 │   │   │   return inner_training_loop(                                   